# Loudness and intermittency

Run from the repository root after building the database from the prepared data bundle; see README.md.


# Initialization and database connection

In [ ]:
from analysis_paths import connect_analysis, data_root, output_path, require_file
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from helpers import plot_sensors_on_map_df
from scipy.stats import pearsonr, spearmanr

# Connect to the prepared DuckDB database
db = connect_analysis()


print("DuckDB version:", db.execute("SELECT version()" ).fetchone()[0])

# Sensor locations

In [ ]:
# Dynamic map with sensor IDs using Plotly Express

sensors_df = db.sql("""
  SELECT sensor_id, latitude, longitude
  FROM sensors
  WHERE latitude IS NOT NULL AND longitude IS NOT NULL
""").df()

def last2(sid):
  return str(sid)[-2:]

sensors_df['label'] = sensors_df['sensor_id'].apply(last2)

fig = px.scatter_map(
  sensors_df,
  lat="latitude",
  lon="longitude",
  hover_name="sensor_id",
  text="label",
  zoom=13,
  width=1200,
  height=800,
  color_discrete_sequence=["#d62728"],
  title="Sensors",
)
fig.update_traces(marker=dict(size=15, opacity=0.8), textposition="top center", textfont=dict(color="#850000", size=12))
fig.update_layout(mapbox_center_lat=sensors_df['latitude'].astype(float).mean(), mapbox_center_lon=sensors_df['longitude'].astype(float).mean())
fig.update_layout(margin={"r":0,"t":35,"l":0,"b":0})
layout_kwargs = {
  "map": {
    "style": "open-street-map",
    "zoom": 13,
  },

}
fig.update_layout(**layout_kwargs)
config = {
  'toImageButtonOptions': {
  'format': 'png', # one of png, svg, jpeg, webp
  }
}
fig.show(config=config)

# Sampling ratios

In [ ]:
# Sampling ratio (from loudness table) based on 8 loudness measurements per second.
sampling_ratio_per_day = db.sql(
  """
  WITH per_day AS (
    SELECT
    sensor_id,
    CAST(time AS DATE) AS date,
    COUNT(*) AS n_samples,
    MIN(time) AS t0,
    MAX(time) AS t1,
    (epoch(MAX(time)) - epoch(MIN(time))) AS span_seconds
    FROM loudness
    GROUP BY 1, 2
  )
  SELECT
    sensor_id,
    date,
    n_samples,
    span_seconds,
    8.0 * span_seconds AS expected_samples,
    n_samples / NULLIF(8.0 * span_seconds, 0) AS sampling_ratio
  FROM per_day
  ORDER BY sensor_id, date
  """
).df()

sampling_ratio_per_day.sort_values('sampling_ratio', ascending=True)

In [ ]:
# Average sampling ratio per sensor
avg_sampling_ratio_per_sensor = sampling_ratio_per_day.groupby('sensor_id')['sampling_ratio'].mean().reset_index()
avg_sampling_ratio_per_sensor = avg_sampling_ratio_per_sensor.sort_values('sampling_ratio', ascending=True)
avg_sampling_ratio_per_sensor

In [ ]:
# Mean average sampling ratio across all sensors
print(avg_sampling_ratio_per_sensor['sampling_ratio'].mean() * 100)

# Loudness

In [ ]:
# Plot avg_dba_per_hour as individual bar chart per sensor in a grid
avg_spla_per_hour = db.sql("SELECT sensor_id, hour, avg_dba, avg_spla FROM loudness_per_hour").df()
selected_sensors = avg_spla_per_hour['sensor_id'].unique()
n_sensors = len(selected_sensors)
n_cols = 8
n_rows = int(np.ceil(n_sensors / n_cols))

# Define hours as strings '00' to '23'
hours = [f"{h:02d}" for h in range(24)]

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3), sharey=True) # , dpi=200
axes = axes.flatten()

for i, sensor_id in enumerate(selected_sensors):
  ax = axes[i]
  df_survey = avg_spla_per_hour[avg_spla_per_hour['sensor_id'] == sensor_id]
  df_survey = df_survey.set_index('hour').reindex(hours, fill_value=np.nan)
  ax.bar(hours, df_survey['avg_dba'], color='skyblue')
  ax.set_title(sensor_id, fontsize=10)
  ax.set_xticks(hours[::2])
  ax.set_xticklabels(hours[::2], fontsize=8, rotation=45)
  ax.set_ylim(30, 70)
  ax.grid(True, axis='y', linestyle='--', alpha=0.5)

# Hide unused axes
for j in range(i+1, len(axes)):
  axes[j].set_visible(False)

fig.suptitle("Average dBA per Hour", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Plot median_loudness_per_hour as individual bar chart per sensor in a grid
median_spla_per_hour = db.sql("SELECT sensor_id, hour, median_dba FROM loudness_per_hour").df()
selected_sensors = median_spla_per_hour['sensor_id'].unique()
n_sensors = len(selected_sensors)
n_cols = 8
n_rows = int(np.ceil(n_sensors / n_cols))

# Define hours as strings '00' to '23'
hours = [f"{h:02d}" for h in range(24)]

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3.5, n_rows*3), sharey=True) #, dpi=200
axes = axes.flatten()

for i, sensor_id in enumerate(selected_sensors):
  ax = axes[i]
  df_survey = median_spla_per_hour[median_spla_per_hour['sensor_id'] == sensor_id]
  df_survey = df_survey.set_index('hour').reindex(hours, fill_value=np.nan)
  ax.bar(hours, df_survey['median_dba'], color='skyblue')
  ax.set_xticks(hours[::2])
  ax.set_xticklabels(hours[::2], fontsize=8, rotation=45)
  ax.set_ylim(30, 70)
  ax.grid(True, axis='y', linestyle='--', alpha=0.5)

# Hide unused axes
for j in range(i+1, len(axes)):
  axes[j].set_visible(False)

fig.suptitle("Median dBA per hour per sensor", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Loudness map

In [ ]:
# Compute mean avg_spla (linear) for each sensor from hourly data, then convert to dB

mean_spla = avg_spla_per_hour.groupby('sensor_id')['avg_spla'].mean().reset_index()

mean_spla['avg_dba'] = 10 * np.log10(mean_spla['avg_spla'])

# Sort by loudness
mean_spla = mean_spla.sort_values('avg_dba', ascending=True)

# Plot and color dots by converted dB value; hover shows both columns
plot_sensors_on_map_df(
  mean_spla,
  sensor_id_col='sensor_id',
  title="Loudness per Sensor",
  color_col='avg_dba',
  color_scale='Turbo',
  dot_size=25,
  width=1600,
  height=800,
)

## Examples for overall loudness and strength of day-night cycle

In [ ]:
median_spla_per_hour = db.sql("SELECT sensor_id, hour, median_dba FROM loudness_per_hour").df()
selected_sensors = ['citizen-007', 'citizen-030', 'citizen-006', 'citizen-005']
n_sensors = len(selected_sensors)
n_cols = 2
n_rows = int(np.ceil(n_sensors / n_cols))

# Define hours as strings '00' to '23'
hours = [f"{h:02d}" for h in range(24)]

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*3), sharey=True) # , dpi=200
axes = axes.flatten()

for i, sensor_id in enumerate(selected_sensors):
  ax = axes[i]
  df_survey = median_spla_per_hour[median_spla_per_hour['sensor_id'] == sensor_id]
  df_survey = df_survey.set_index('hour').reindex(hours, fill_value=np.nan)
  ax.bar(hours, df_survey['median_dba'], color='skyblue')
  ax.set_title(sensor_id, fontsize=10)
  ax.set_xticks(hours[::2])
  ax.set_xticklabels(hours[::2], fontsize=8, rotation=45)
  ax.set_ylim(30, 70)
  ax.grid(True, axis='y', linestyle='--', alpha=0.5)

# Hide unused axes
for j in range(i+1, len(axes)):
  axes[j].set_visible(False)

fig.suptitle("Median dBA per hour per sensor", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
median_spla_per_hour = db.sql("SELECT sensor_id, hour, median_spla FROM loudness_per_hour").df()
mean_spla_per_hour = db.sql("SELECT hour, avg_spla FROM loudness_per_hour").df()
median_hourly = median_spla_per_hour.groupby('hour')['median_spla'].median().reindex(hours)
mean_hourly = mean_spla_per_hour.groupby('hour')['avg_spla'].mean().reindex(hours)
median_hourly_dba = 10 * np.log10(median_hourly)
mean_hourly_dba = 10 * np.log10(mean_hourly)

plt.figure(figsize=(8, 3))
plt.bar(hours, median_hourly_dba, color='skyblue', label='Median')
plt.plot(hours, mean_hourly_dba, color='orange', marker='o', label='Mean')
plt.title("Median and mean hourly loudness over all sensors")
plt.xlabel("Time of Day")
plt.ylabel("Loudness in dBA")
plt.xticks(hours[::2])
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.ylim(34.95, 65)
plt.legend()
plt.tight_layout()
plt.show()

# Intermittency Ratio

In [ ]:
C = 10 # Threshold offset in dB

# Intermittency ratio per sensor per hour
ir_per_hour = db.sql("SELECT * from ir_per_hour").df()
display(ir_per_hour.head())

In [ ]:
# Plot histogram of intermittency ratio for each sensor as subplots in a grid
sensor_ids = ir_per_hour['sensor_id'].unique()
n_sensors = len(sensor_ids)
n_cols = 8
n_rows = int(np.ceil(n_sensors / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*1.75, n_rows*1.75), sharey=True)
axes = axes.flatten()

for i, sensor_id in enumerate(sensor_ids):
  ax = axes[i]
  data = ir_per_hour[ir_per_hour['sensor_id'] == sensor_id]['intermittency_ratio']
  # Plot as percentages instead of absolute counts
  counts, bins = np.histogram(data, bins=10, range=(0, 100))
  percents = 100 * counts / counts.sum() if counts.sum() > 0 else counts
  ax.bar(bins[:-1], percents, width=(bins[1] - bins[0]), color='skyblue', edgecolor='black', align='edge')
  ax.set_ylim(0, 100)
  ax.set_yticks([0, 25, 50, 75, 100])
  ax.set_ylabel('Percent' if i % n_cols == 0 else '')
  ax.set_title(sensor_id, fontsize=10)
  ax.set_xlim(0, 100)
  ax.set_xticks([0, 25, 50, 75, 100])
  ax.grid(axis='y', linestyle='--', alpha=0.5)
  if i % n_cols == 0:
    ax.set_ylabel('Frequency (%)')
  if i >= (n_rows - 1) * n_cols:
    ax.set_xlabel('Intermittency Ratio')

# Hide unused axes
for j in range(i+1, len(axes)):
  axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Intermittency Ratio map

In [ ]:
# Average IR per sensor
ir_per_sensor = ir_per_hour.groupby('sensor_id')['intermittency_ratio'].mean().reset_index().sort_values('sensor_id', ascending=False)

plot_sensors_on_map_df(
  ir_per_sensor,
  sensor_id_col='sensor_id',
  title="Intermittency Ratio",
  color_col='intermittency_ratio',
  color_scale='Turbo',
  dot_size=25,
  width=1920,
  height=1000,
)

## Intermittency vs Loudness

In [ ]:
loudness_df = db.sql("""
  SELECT sensor_id, 10*log10(avg(avg_spla)) AS dba
  FROM loudness_per_hour
  GROUP BY sensor_id
""").df()

scatter_df = loudness_df.merge(ir_per_sensor, on='sensor_id', how='inner')

fig, ax = plt.subplots(figsize=(5, 3), dpi=200)
ax.scatter(scatter_df['dba'], scatter_df['intermittency_ratio'], s=30, alpha=0.8)
ax.set_xlabel(r'$L_{\mathrm{Aeq}}$ (dB(A))')
ax.set_ylabel('Average Intermittency Ratio (%)')
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_ylim(0, max(scatter_df['intermittency_ratio']) + 5)
plt.tight_layout()
plt.show()

corr_df = scatter_df[['dba', 'intermittency_ratio']].dropna()

pearson_r, pearson_p = pearsonr(corr_df['dba'], corr_df['intermittency_ratio'])
spearman_rho, spearman_p = spearmanr(corr_df['dba'], corr_df['intermittency_ratio'])

print(f'n = {len(corr_df)} sensors')
print(f'Pearson r = {pearson_r:.3f}, p = {pearson_p:.4f}')
print(f'Spearman rho = {spearman_rho:.3f}, p = {spearman_p:.4f}')

In [ ]:
ir_per_sensor

# Causes of Intermittency

In [ ]:
from tqdm import tqdm

C = 10  # threshold offset in dB
PROB_THRESHOLD = 0.5
labels = [
  "vehicle",
  "honking",
  "aircraft",
  "siren",
  "human",
  "bark",
  "bird",
  "church",
  "music",
  "wind",
  "rain",
]
MIN_SAMPLES_PER_HOUR = 15 * 20  # at least 15 minutes of 3s frames (~20 frames/min)

def intermittent_class_counts_per_hour(
  g: pd.DataFrame,
  C: float,
  class_cols: list[str],
  prob_threshold: float = 0.5,
  min_samples: int = MIN_SAMPLES_PER_HOUR,
 ) -> dict[str, int] | None:
  if len(g) < min_samples:
    return None

  # Leq_tot computed correctly in dB from linear avg_spl_a
  Leq_tot = 10 * np.log10(np.mean(g["avg_spl_a"]))
  K = Leq_tot + C

  avg_dba = 10 * np.log10(g["avg_spl_a"])
  in_events = g.loc[avg_dba > K, class_cols]
  if in_events.empty:
    return None

  # per-label counts for this hour
  return (in_events >= prob_threshold).sum(axis=0).astype(int).to_dict()

def _query_sources_for_sensor(sensor_id: str, class_cols: list[str]) -> pd.DataFrame:
  class_select = ", ".join([f'"{c}"' for c in class_cols])
  return db.execute(
    f"""
    SELECT
      date_trunc('hour', time) AS hour,
      avg_spl_a,
      {class_select}
    FROM sources
    WHERE sensor_id = ? AND avg_spl_a > 0
    """,
    [sensor_id],
  ).df()

sensor_ids = db.sql("SELECT DISTINCT sensor_id FROM sources ORDER BY sensor_id").df()["sensor_id"].tolist()

rows: list[dict] = []
for sid in tqdm(sensor_ids):
  df = _query_sources_for_sensor(sid, labels)
  if df.empty:
    continue

  df["hour"] = pd.to_datetime(df["hour"])

  for hour, g in df.groupby("hour", sort=True):
    counts = intermittent_class_counts_per_hour(
      g, C=C, class_cols=labels, prob_threshold=PROB_THRESHOLD
    )
    if not counts:
      continue

    rows.append(
      {
        "sensor_id": sid,
        "hour_of_day": int(hour.hour),
        **counts,
      }
    )

intermittent_event_counts = pd.DataFrame(rows)
intermittent_event_counts = intermittent_event_counts.groupby(["sensor_id", "hour_of_day"], as_index=False)[labels].sum()
intermittent_event_counts[labels] = intermittent_event_counts[labels].astype("int64")
intermittent_event_counts = intermittent_event_counts.sort_values(["sensor_id", "hour_of_day"]).reset_index(drop=True)

In [ ]:
# Sum counts per sensor and find the dominant source class
event_sums_per_sensor = intermittent_event_counts.groupby('sensor_id')[labels].sum().reset_index()
event_sums_per_sensor['dominant_source'] = event_sums_per_sensor[labels].idxmax(axis=1)
event_sums_per_sensor.head(5)

The derived `intermittent_event_sums_per_sensor` table is supplied by the prepared database. Analysis stays read-only; the preceding cells recompute the counts for inspection without overwriting that table.


In [ ]:
# Plot total counts over all sensors as a bar chart
total_ir_causes = event_sums_per_sensor.groupby('sensor_id').sum()[labels].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(total_ir_causes.index, total_ir_causes.values)
plt.xlabel('Sound event')
plt.ylabel('Total Count of intermittent events')
plt.title('Total Intermittent Event Counts Across All Sensors')
plt.xticks(rotation=45, ha='right')
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Dominant cause of intermittency

In [ ]:
sensors_df = db.sql("""
    SELECT sensor_id, latitude, longitude
    FROM sensors
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL
""").df()

most_common_loc = event_sums_per_sensor.merge(sensors_df, on='sensor_id', how='left')

fig = px.scatter_map(
    most_common_loc,
    lat="latitude",
    lon="longitude",
    color="dominant_source",   # different color per sound source
    hover_name="sensor_id",
    zoom=11,
    height=700,
    width=1200
)
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0)
)
fig.update_traces(
    marker=dict(size=15)   # adjust number as desired
)

fig.show()

In [ ]:
most_common_loc

## Most common dominant causes of intermittency

In [ ]:
sensors_df = db.sql("""
    SELECT sensor_id, latitude, longitude
    FROM sensors
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL
""").df()

most_common_loc = event_sums_per_sensor.merge(sensors_df, on='sensor_id', how='left')

keep_main = ["vehicle", "human", "bird"]
exclude_from_other = set(keep_main + ["wind", "rain", "aircraft"])

# all source columns available in most_common_loc (based on `labels` from earlier cells)
source_cols = [c for c in labels if c in most_common_loc.columns]

other_cols = [c for c in source_cols if c not in exclude_from_other]

most_common_loc_grouped = most_common_loc[["sensor_id", "latitude", "longitude"] + keep_main].copy()
most_common_loc_grouped["other"] = most_common_loc[other_cols].sum(axis=1)

# dominant within the 4 groups only
most_common_loc_grouped["dominant_source"] = most_common_loc_grouped[keep_main + ["other"]].idxmax(axis=1)

color_map = {
    "vehicle": "#c92121",
    "human":   "#4abdcc",
    "bird":    "#2ca02c",
    "other":   "#585858",
}
legend_order = list(color_map.keys())

fig = px.scatter_map(
    most_common_loc_grouped,
    lat="latitude",
    lon="longitude",
    color="dominant_source",   # different color per sound source
    hover_name="sensor_id",
    zoom=11,
    height=700,
    width=1200,
    color_discrete_map=color_map,
    category_orders={"dominant_source": legend_order},
 )
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0)
 )
fig.update_traces(
    marker=dict(size=18)   # adjust number as desired
 )

fig.show()

In [ ]:
import base64
from io import BytesIO
import plotly.express as px
from pathlib import Path
from PIL import Image

keep_main = ["vehicle", "human", "bird"]
exclude_from_other = set(keep_main + ["wind", "rain", "aircraft"])

# all source columns available in most_common_loc (based on `labels` from earlier cells)
source_cols = [c for c in labels if c in most_common_loc.columns]

other_cols = [c for c in source_cols if c not in exclude_from_other]

most_common_loc_pie = most_common_loc[["sensor_id", "latitude", "longitude"] + keep_main].copy()
most_common_loc_pie["other"] = most_common_loc[other_cols].sum(axis=1)

# dominant within the main groups only
most_common_loc_pie["dominant_source"] = most_common_loc_pie[keep_main + ["other"]].idxmax(axis=1)


pie_cols = ["vehicle", "human", "bird", "other"]

pie_colors = {
    "vehicle": "#0072B2",
    "human": "#E69F00",
    "bird": "#009E73",
    "other": "#999999",
}

base = px.scatter_map(
    most_common_loc_pie,
    lat="latitude",
    lon="longitude",
    hover_name="sensor_id",
    hover_data=pie_cols,
    zoom=11,
    height=800,
    width=1200,
 )
base.update_traces(marker=dict(size=1, opacity=0))
base.update_layout(margin=dict(l=0, r=0, t=0, b=0), showlegend=True)

def _trim_transparent_png(png_bytes: bytes) -> bytes:
    """Crop away fully-transparent border pixels (requires Pillow)."""
    if Image is None:
        return png_bytes
    im = Image.open(BytesIO(png_bytes)).convert("RGBA")
    alpha = im.split()[-1]
    bbox = alpha.getbbox()
    if bbox is None:
        return png_bytes
    im = im.crop(bbox)
    out = BytesIO()
    im.save(out, format="PNG")
    return out.getvalue()

def _pie_png(values):
    fig, ax = plt.subplots(figsize=(1.0, 1.0), dpi=160)
    ax.pie(values, startangle=90, colors=[pie_colors[c] for c in pie_cols])
    ax.set(aspect="equal")
    ax.axis("off")
    ax.set_position([0, 0, 1, 1])
    buf = BytesIO()
    fig.savefig(
        buf,
        format="png",
        transparent=True,
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.close(fig)
    png_bytes = _trim_transparent_png(buf.getvalue())
    return "data:image/png;base64," + base64.b64encode(png_bytes).decode()

d_lat = 0.0006  # degrees latitude
layers = []
for r in most_common_loc_pie.itertuples(index=False):
    lon, lat = float(r.longitude), float(r.latitude)
    d_lon = d_lat / np.cos(np.deg2rad(lat))
    src = _pie_png([getattr(r, c) for c in pie_cols])
    layers.append(
        dict(
            sourcetype="image",
            source=src,
            coordinates=[[lon - d_lon, lat + d_lat], [lon + d_lon, lat + d_lat], [lon + d_lon, lat - d_lat], [lon - d_lon, lat - d_lat]],
        )
    )

# Legend entries for slice colors
lat0 = float(most_common_loc_pie["latitude"].iloc[0])
lon0 = float(most_common_loc_pie["longitude"].iloc[0])
for name, color in pie_colors.items():
    base.add_trace(
        dict(
            type="scattermap",
            lat=[lat0],
            lon=[lon0],
            mode="markers",
            marker=dict(size=12, color=color),
            name=name,
            visible="legendonly",
            showlegend=True,
        )
    )

base.update_layout(map=dict(style="open-street-map", layers=layers))
base.show()

# Save bar charts to folder.
def _bar_png(values, normalize: bool = True):
    values = np.asarray(values, dtype=float)
    if normalize:
        s = values.sum()
        values = values / s if s > 0 else values
    fig, ax = plt.subplots(figsize=(1.8, 0.4), dpi=160)
    x = np.arange(len(pie_cols))
    ax.bar(x, values, color=[pie_colors[c] for c in pie_cols], width=0.85)
    ax.set_xlim(-0.5, len(pie_cols) - 0.5)
    ax.set_ylim(0, 1.0 if normalize else (values.max() * 1.05 if values.size else 1.0))
    ax.axis("off")
    ax.set_position([0, 0, 1, 1])
    buf = BytesIO()
    fig.savefig(
        buf,
        format="png",
        transparent=True,
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.close(fig)
    png_bytes = buf.getvalue()
    return "data:image/png;base64," + base64.b64encode(png_bytes).decode(), png_bytes

out_dir = output_path("figures/ir_cause_bars")
out_dir.mkdir(parents=True, exist_ok=True)

for r in most_common_loc_pie.itertuples(index=False):
    _, png_bytes = _bar_png([getattr(r, c) for c in pie_cols], normalize=True)
    (out_dir / f"{r.sensor_id}.png").write_bytes(png_bytes)

print(f"Saved {len(most_common_loc_pie)} bars to {out_dir}/")

In [ ]:
# Obtain relative numbers
most_common_loc_pie_rel = most_common_loc_pie.copy()
most_common_loc_pie_rel[pie_cols] = most_common_loc_pie_rel[pie_cols].div(most_common_loc_pie_rel[pie_cols].sum(axis=1), axis=0)

# Weight by IR contribution per sensor
ir_per_sensor = ir_per_hour.groupby('sensor_id')['intermittency_ratio'].mean()
most_common_loc_pie_rel = most_common_loc_pie_rel.merge(ir_per_sensor, on='sensor_id', how='left')
most_common_loc_pie_rel[pie_cols] = most_common_loc_pie_rel[pie_cols].multiply(most_common_loc_pie_rel['intermittency_ratio'] / 100, axis=0)

# Average relative contributions of each source across all sensors to get the overall distribution
rel_sum = most_common_loc_pie_rel[pie_cols].sum()
print(rel_sum)
_, png_bytes = _bar_png([getattr(rel_sum, c) for c in pie_cols], normalize=True)
(out_dir / f"overall.png").write_bytes(png_bytes)

import IPython.display
IPython.display.Image(data=png_bytes)